# 📈 Proyecto VaR — Exploración y Análisis

Este notebook muestra paso a paso cómo funciona el pipeline de datos.
Para la interfaz completa, ejecuta `python src/app.py`.

**Estructura:**
1. Descarga de datos (Bronze)
2. Limpieza y cálculo de retornos (Silver)
3. Cálculo del VaR (Gold)
4. Visualizaciones

In [ ]:
# Importaciones necesarias
import sys
import os
sys.path.insert(0, os.path.join('..', 'src'))  # agregar src al path

import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
import plotly.graph_objects as go
import plotly.express as px

print('✅ Librerías importadas correctamente')

## 1. 🥉 BRONZE — Descarga de datos crudos

In [ ]:
# Definir el portafolio
tickers = ['AAPL', 'MSFT', 'GOOGL']
periodo_dias = 365

fecha_fin = datetime.today()
fecha_inicio = fecha_fin - timedelta(days=periodo_dias)

print(f'Descargando datos desde {fecha_inicio.date()} hasta {fecha_fin.date()}')

todos = []
for ticker in tickers:
    datos = yf.download(ticker, start=fecha_inicio.strftime('%Y-%m-%d'),
                        end=fecha_fin.strftime('%Y-%m-%d'), progress=False, auto_adjust=True)
    datos = datos[['Close']].copy()
    datos.columns = ['close']
    datos['ticker'] = ticker
    datos = datos.reset_index().rename(columns={'Date': 'fecha'})
    todos.append(datos)
    print(f'  ✓ {ticker}: {len(datos)} registros')

df_bronze = pd.concat(todos, ignore_index=True)
print(f'\nTotal registros Bronze: {len(df_bronze)}')
df_bronze.head(10)

## 2. 🥈 SILVER — Limpieza y retornos logarítmicos

In [ ]:
df_silver = df_bronze.copy()
df_silver['fecha'] = pd.to_datetime(df_silver['fecha'])
df_silver = df_silver.sort_values(['ticker', 'fecha'])
df_silver['close'] = pd.to_numeric(df_silver['close'], errors='coerce')
df_silver = df_silver.dropna(subset=['close'])

# Retorno logarítmico: ln(P_hoy / P_ayer)
df_silver['retorno_log'] = df_silver.groupby('ticker')['close'].transform(
    lambda x: np.log(x / x.shift(1))
)
df_silver = df_silver.dropna(subset=['retorno_log'])

print(f'Registros Silver (con retornos): {len(df_silver)}')
print('\nEstadísticas de retornos por activo:')
df_silver.groupby('ticker')['retorno_log'].describe().round(4)

## 3. 🥇 GOLD — Cálculo del VaR

In [ ]:
# Parámetros del portafolio
pesos = {'AAPL': 0.4, 'MSFT': 0.35, 'GOOGL': 0.25}
nivel_confianza = 0.95
valor_portafolio = 100_000  # USD

# Pivotear: filas=fecha, columnas=ticker
tabla = df_silver.pivot_table(index='fecha', columns='ticker', values='retorno_log')
tabla = tabla.dropna()

# Vector de pesos (normalizado)
tickers_ord = list(pesos.keys())
pesos_array = np.array([pesos[t] for t in tickers_ord])
pesos_array = pesos_array / pesos_array.sum()

# Retorno ponderado del portafolio
retornos_portafolio = tabla[tickers_ord].values @ pesos_array

# VaR al percentil 5 (para confianza 95%)
var_pct = np.percentile(retornos_portafolio, 5)
var_dinero = abs(var_pct) * valor_portafolio

# CVaR: promedio de las pérdidas peores que el VaR
cvar_pct = retornos_portafolio[retornos_portafolio <= var_pct].mean()
cvar_dinero = abs(cvar_pct) * valor_portafolio

print(f'📊 RESULTADOS DEL PORTAFOLIO')
print(f'Valor del portafolio: ${valor_portafolio:,.0f} USD')
print(f'Nivel de confianza: {nivel_confianza:.0%}')
print(f'Observaciones: {len(retornos_portafolio)} días')
print()
print(f'VaR (95%)  = {abs(var_pct):.4%}  →  ${var_dinero:,.2f} USD')
print(f'CVaR       = {abs(cvar_pct):.4%}  →  ${cvar_dinero:,.2f} USD')
print()
print(f'💡 Interpretación: Con {nivel_confianza:.0%} de confianza, el portafolio')
print(f'   NO perderá más de ${var_dinero:,.2f} en un día de mercado.')

## 4. 📊 Visualización — Distribución de retornos con VaR

In [ ]:
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=retornos_portafolio,
    nbinsx=50,
    name='Retornos del portafolio',
    marker_color='#4C72B0',
    opacity=0.75
))

# Zona de pérdida
x_loss = retornos_portafolio[retornos_portafolio <= var_pct]
fig.add_trace(go.Histogram(
    x=x_loss,
    nbinsx=15,
    name='Zona de pérdida (5%)',
    marker_color='red',
    opacity=0.6
))

fig.add_vline(x=var_pct, line_color='red', line_dash='dash',
              annotation_text=f'VaR 95% = {var_pct:.2%}', annotation_font_color='red')

fig.update_layout(
    title='Distribución de Retornos del Portafolio',
    xaxis_title='Retorno Diario',
    yaxis_title='Frecuencia',
    xaxis_tickformat='.1%',
    template='plotly_white',
    height=400
)

fig.show()

## 5. 🚀 Ejecutar la App completa

Para ver la interfaz gráfica completa con todos los gráficos y controles interactivos, ejecuta en tu terminal:

```bash
python src/app.py
```

Y abre en tu navegador: **http://localhost:7860**